In [9]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import os

# 1. CONFIGURATION


In [12]:


EXCEL_FILE = r"TechCorp_PowerBI_Dataset.xlsx"

SERVER = r"localhost"
DATABASE = "TechCorp_ProjectManagement_DB"
DRIVER = "ODBC Driver 18 for SQL Server"

# Windows Authentication
connection_string = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection_string)
)

# 2. EXCEL SHEET → SQL STAGING TABLE MAPPING

In [13]:
sheet_table_map = {
    "Employees": "Employees",
    "Projects": "Projects",
    "Project_Assignments": "ProjectAssignments",
    "Tasks": "Tasks",
    "Performance_Reviews": "PerformanceReviews",
    "Milestones": "Milestones"
}

# 3. TEST SQL SERVER CONNECTION

In [14]:
print("\nTesting SQL Server connection...")

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@SERVERNAME"))
        server_name = result.scalar()

    print(f"Connected successfully to: {server_name}")
    print(f"Database: {DATABASE}")

except Exception as e:
    print("\nSQL Server connection failed.")
    print(e)
    raise SystemExit()



Testing SQL Server connection...


C:\Users\iroge\AppData\Local\Temp\ipykernel_20128\1184783544.py:4: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Connected successfully to: COVENANT_OFGOD
Database: TechCorp_ProjectManagement_DB


# 4. LOAD EACH EXCEL SHEET

In [18]:
print("\nStarting ETL...\n")

row_counts = {}

for sheet_name, table_name in sheet_table_map.items():

    print(f"Reading sheet: {sheet_name}")

    # Read Excel
    df = pd.read_excel(
        EXCEL_FILE,
        sheet_name=sheet_name
    )
    # --------------------------------------------------------
    # Preserve NULL values
    # --------------------------------------------------------

    df = df.where(pd.notna(df), None)

    # --------------------------------------------------------
    # Convert staging values to strings
    # --------------------------------------------------------
    # Staging is intentionally raw.
    # We will perform proper type conversion later
    # when moving data into dbo tables.

    for column in df.columns:
        df[column] = df[column].apply(
            lambda x: None if pd.isna(x) else str(x)
        )

    # --------------------------------------------------------
    # Get row count
    # --------------------------------------------------------

    excel_count = len(df)

    print(f"  Rows found: {excel_count}")

    # --------------------------------------------------------
    # Clear existing staging data
    # --------------------------------------------------------

    with engine.begin() as conn:
        conn.execute(
            text(f"DELETE FROM stg.[{table_name}]")
        )

    # --------------------------------------------------------
    # Load into SQL Server staging
    # --------------------------------------------------------

    df.to_sql(
        name=table_name,
        con=engine,
        schema="stg",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    row_counts[sheet_name] = excel_count

    print(f"  Loaded into: stg.{table_name}")
    print()



Starting ETL...

Reading sheet: Employees
  Rows found: 30
  Loaded into: stg.Employees

Reading sheet: Projects
  Rows found: 15
  Loaded into: stg.Projects

Reading sheet: Project_Assignments
  Rows found: 78
  Loaded into: stg.ProjectAssignments

Reading sheet: Tasks
  Rows found: 189
  Loaded into: stg.Tasks

Reading sheet: Performance_Reviews
  Rows found: 60
  Loaded into: stg.PerformanceReviews

Reading sheet: Milestones
  Rows found: 65
  Loaded into: stg.Milestones



# 5. VALIDATE SQL ROW COUNTS

In [19]:
print("=" * 60)
print("ROW COUNT VALIDATION")
print("=" * 60)

with engine.connect() as conn:

    for sheet_name, table_name in sheet_table_map.items():

        result = conn.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM stg.[{table_name}]"
            )
        )

        sql_count = result.scalar()
        excel_count = row_counts[sheet_name]

        status = "PASS" if sql_count == excel_count else "FAIL"

        print(
            f"{sheet_name:<25} "
            f"Excel: {excel_count:<5} "
            f"SQL: {sql_count:<5} "
            f"[{status}]"
        )


ROW COUNT VALIDATION
Employees                 Excel: 30    SQL: 30    [PASS]
Projects                  Excel: 15    SQL: 15    [PASS]
Project_Assignments       Excel: 78    SQL: 78    [PASS]
Tasks                     Excel: 189   SQL: 189   [PASS]
Performance_Reviews       Excel: 60    SQL: 60    [PASS]
Milestones                Excel: 65    SQL: 65    [PASS]


# 6. FINISHED

In [20]:
print("\nETL process completed.")


ETL process completed.
